In [ ]:
from pathlib import Path
import sys

# Add packages to sys.path to allow imports
sys.path.append(str(Path.cwd().parent / 'packages' / 'infra' / 'src'))
sys.path.append(str(Path.cwd().parent / 'packages' / 'core' / 'src'))

from algotrading.infra.storage.adapter import ParquetStore

# The project root is one level up from the notebooks directory
repo_root = Path('..')
store_root = repo_root / 'data'

print(f"Using data store root: {store_root.resolve()}")

store = ParquetStore(store_root)

try:
    daily_bars = store.read('daily_bar')
    
    if not daily_bars:
        print("No data found in 'daily_bar' table.")
    else:
        print(f"Successfully loaded {len(daily_bars)} records from 'daily_bar'.")
        # Store the loaded bars for the next cell
        %store daily_bars
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import datetime
import plotly.graph_objects as go

# Restore the daily_bars variable from the previous cell
%store -r daily_bars

if 'daily_bars' in locals() and daily_bars:
    # Filter for a specific underlying and time period
    underlying_to_plot = 'BAC'
    start_date = datetime.date(2023, 1, 1)

    filtered_bars = [
        bar for bar in daily_bars 
        if bar.underlying == underlying_to_plot and bar.trade_date >= start_date
    ]

    if not filtered_bars:
        print(f"No data found for {underlying_to_plot} after {start_date}")
    else:
        print(f"Plotting {len(filtered_bars)} bars for {underlying_to_plot}")
        fig = go.Figure(data=[go.Candlestick(
            x=[bar.trade_date for bar in filtered_bars],
            open=[bar.open for bar in filtered_bars],
            high=[bar.high for bar in filtered_bars],
            low=[bar.low for bar in filtered_bars],
            close=[bar.close for bar in filtered_bars]
        )])
        
        fig.update_layout(
            title=f'{underlying_to_plot} Candlestick Chart (from {start_date})',
            xaxis_title='Date',
            yaxis_title='Price'
        )
        
        # Save the figure
        output_image_path = 'historical_data_plot.png'
        fig.write_image(output_image_path)
        print(f"Figure saved to {output_image_path}")
        fig.show()
else:
    print("daily_bars variable not found. Please run the first cell.")